In [1]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

# install deps
!pip install -q pandas requests

import os

base_dir = "/content/drive/MyDrive/Bluesky"
print("drive mounted")
print(os.listdir(base_dir))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
drive mounted
['bluesky_politician_scraper.ipynb', 'nonmilletvekili.json', 'milletvekili.json']


In [2]:
import os
import json
import pandas as pd

# set base dir on drive
base_dir = "/content/drive/MyDrive/Bluesky"
mv_path = os.path.join(base_dir, "milletvekili.json")
nonmv_path = os.path.join(base_dir, "nonmilletvekili.json")

# helper to load json into dataframe
def load_json_to_df(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    # root can be list or dict-with-list
    if isinstance(data, list):
        records = data
    elif isinstance(data, dict):
        list_key = None
        for k, v in data.items():
            if isinstance(v, list):
                list_key = k
                break
        if list_key is None:
            raise ValueError("no list field found in json")
        records = data[list_key]
    else:
        raise ValueError("unexpected json root type")
    return pd.DataFrame(records)

# load both jsons
mv_df = load_json_to_df(mv_path)
nonmv_df = load_json_to_df(nonmv_path)

# add isMilletvekili flag
mv_df["isMilletvekili"] = True
nonmv_df["isMilletvekili"] = False

# ensure common columns exist
for col in ["district", "status_change"]:
    if col not in nonmv_df.columns:
        nonmv_df[col] = None

for col in ["profession", "title", "platform", "political_stance"]:
    if col not in mv_df.columns:
        mv_df[col] = None
    if col not in nonmv_df.columns:
        nonmv_df[col] = None

# combine into single df
df = pd.concat([mv_df, nonmv_df], ignore_index=True)

# fill role info for deputies
mask_mv = df["isMilletvekili"] == True
df.loc[mask_mv, "profession"] = df.loc[mask_mv, "profession"].fillna("Milletvekili")
df.loc[mask_mv, "title"] = df.loc[mask_mv, "title"].fillna("TBMM Milletvekili")
df.loc[mask_mv, "platform"] = df.loc[mask_mv, "platform"].fillna("TBMM")

# infer political stance from party/alliance
def infer_stance(party, alliance):
    if party is None:
        return "Bağımsız / Belirtilmemiş"

    # iktidar kanadı
    if party == "Adalet ve Kalkınma Partisi":
        return "Muhafazakar İktidar"
    if party == "Milliyetçi Hareket Partisi":
        return "Milliyetçi İktidar"
    if party == "Yeniden Refah Partisi":
        return "İslamcı Muhafazakar"
    if party == "Hür Dava Partisi" or party == "HÜDA PAR":
        return "İslamcı Kürt Siyaseti"

    # sol sosyal demokrat muhalefet
    if party == "Cumhuriyet Halk Partisi":
        return "Sosyal Demokrat Muhalif"

    # merkez sağ muhalefet
    if party == "İYİ Parti":
        return "Merkez Sağ Milliyetçi Muhalif"

    # kürt siyaseti
    if party in [
        "Halkların Eşitlik ve Demokrasi Partisi",
        "Yeşil Sol Parti",
        "Halkların Demokratik Partisi",
        "Demokratik Bölgeler Partisi",
    ]:
        return "Kürt Siyaseti Sol Muhalif"

    # yeni yol ve benzeri sol partiler
    if party == "Yeni Yol":
        return "Sosyalist Muhalif"

    # bağımsız
    if party == "Bağımsız":
        return "Bağımsız / Belirtilmemiş"

    # fallback
    return "Bağımsız / Belirtilmemiş"

df["political_stance"] = df.apply(
    lambda r: infer_stance(r["party"], r["alliance"]),
    axis=1,
)

print(df.shape)
print(df.head())
print(df["political_stance"].value_counts())


(1200, 12)
   id district      name   surname                       party  \
0   1    Adana      Ömer     Çelik  Adalet ve Kalkınma Partisi   
1   2    Adana     Ahmet  Zenbilci                    Bağımsız   
2   3    Adana     Sunay   Karamık                    Bağımsız   
3   4    Adana  Abdullah     Doğru                    Bağımsız   
4   5    Adana     Faruk     Aytek                    Bağımsız   

          alliance status_change  isMilletvekili    profession  \
0  Cumhur İttifakı          None            True  Milletvekili   
1             None          None            True  Milletvekili   
2             None          None            True  Milletvekili   
3             None          None            True  Milletvekili   
4             None          None            True  Milletvekili   

               title platform          political_stance  
0  TBMM Milletvekili     TBMM       Muhafazakar İktidar  
1  TBMM Milletvekili     TBMM  Bağımsız / Belirtilmemiş  
2  TBMM Milletvekili 

In [13]:
import os
import json
import pandas as pd

# base dir
base_dir = "/content/drive/MyDrive/Bluesky"
mv_path = os.path.join(base_dir, "milletvekili.json")
nonmv_path = os.path.join(base_dir, "nonmilletvekili.json")

# helper to load json into dataframe
def load_json_to_df(path):
    # reads json and returns dataframe from the first list it finds
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        records = data
    elif isinstance(data, dict):
        records = None
        for k, v in data.items():
            if isinstance(v, list):
                records = v
                break
        if records is None:
            raise ValueError("no list found in json")
    else:
        raise ValueError("unexpected json root type")

    return pd.DataFrame(records)

# load jsons
mv_df = load_json_to_df(mv_path)
nonmv_df = load_json_to_df(nonmv_path)

# add isMilletvekili flag
mv_df["isMilletvekili"] = True
nonmv_df["isMilletvekili"] = False

# ensure common columns exist
for col in ["district", "status_change"]:
    if col not in nonmv_df.columns:
        nonmv_df[col] = None

for col in ["profession", "title", "platform", "political_stance"]:
    if col not in mv_df.columns:
        mv_df[col] = None
    if col not in nonmv_df.columns:
        nonmv_df[col] = None

# basic role info for deputies
mask_mv = mv_df["isMilletvekili"] == True
mv_df.loc[mask_mv, "profession"] = mv_df.loc[mask_mv, "profession"].fillna("Milletvekili")
mv_df.loc[mask_mv, "title"] = mv_df.loc[mask_mv, "title"].fillna("TBMM Milletvekili")
mv_df.loc[mask_mv, "platform"] = mv_df.loc[mask_mv, "platform"].fillna("TBMM")

# simple stance mapping (party based, we cannot infer individual ideology objectively)
def infer_stance(party, alliance):
    if party is None:
        return "Bağımsız / Belirtilmemiş"

    if party == "Adalet ve Kalkınma Partisi":
        return "Muhafazakar İktidar"
    if party == "Milliyetçi Hareket Partisi":
        return "Milliyetçi İktidar"
    if party == "Yeniden Refah Partisi":
        return "İslamcı Muhafazakar"
    if party in ["HÜDA PAR", "Hür Dava Partisi"]:
        return "İslamcı Kürt Siyaseti"

    if party == "Cumhuriyet Halk Partisi":
        return "Sosyal Demokrat / Kemalist Muhalif"
    if party == "İYİ Parti":
        return "Merkez Sağ Milliyetçi Muhalif"

    if party in [
        "Halkların Eşitlik ve Demokrasi Partisi",
        "Yeşil Sol Parti",
        "Halkların Demokratik Partisi",
        "Demokratik Bölgeler Partisi",
    ]:
        return "Kürt Siyaseti Sol Muhalif"

    if party == "Yeni Yol":
        return "Sosyalist Muhalif"

    if party == "Bağımsız":
        return "Bağımsız / Belirtilmemiş"

    return "Bağımsız / Belirtilmemiş"

mv_df["political_stance"] = mv_df.apply(
    lambda r: infer_stance(r.get("party"), r.get("alliance")),
    axis=1,
)

# combine
df = pd.concat([mv_df, nonmv_df], ignore_index=True)

out_path = os.path.join(base_dir, "combined_users.csv")
df.to_csv(out_path, index=False)

print("combined_users.csv saved:", out_path)
print("shape:", df.shape)
print(df.head(5))


combined_users.csv saved: /content/drive/MyDrive/Bluesky/combined_users.csv
shape: (1200, 12)
   id district      name   surname                       party  \
0   1    Adana      Ömer     Çelik  Adalet ve Kalkınma Partisi   
1   2    Adana     Ahmet  Zenbilci                    Bağımsız   
2   3    Adana     Sunay   Karamık                    Bağımsız   
3   4    Adana  Abdullah     Doğru                    Bağımsız   
4   5    Adana     Faruk     Aytek                    Bağımsız   

          alliance status_change  isMilletvekili    profession  \
0  Cumhur İttifakı          None            True  Milletvekili   
1             None          None            True  Milletvekili   
2             None          None            True  Milletvekili   
3             None          None            True  Milletvekili   
4             None          None            True  Milletvekili   

               title platform          political_stance  
0  TBMM Milletvekili     TBMM       Muhafazakar İktida

In [20]:
import pandas as pd
import requests
import unicodedata
import time
from tqdm import tqdm

# path to existing csv with previous bluesky search results
csv_path = "/content/drive/MyDrive/Bluesky/combined_users_with_bsky.csv"

# load dataframe
df = pd.read_csv(csv_path)

# ensure bluesky columns exist
for col in ["bsky_handle", "bsky_displayName", "bsky_description", "bsky_match_status", "bsky_best_score"]:
    if col not in df.columns:
        df[col] = pd.NA

# helper: normalize text (remove accents, lowercase)
def normalize_text(s):
    # basic text normalization for turkish names
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return s.lower()

# helper: get short party keyword (akp, chp, mhp, iyi, etc.)
def party_keyword(party_raw):
    # map full party names to short tokens for search
    party = normalize_text(party_raw)
    if not party:
        return ""
    if "adalet ve kalkinma partisi" in party or "ak parti" in party:
        return "akp"
    if "cumhuriyet halk partisi" in party:
        return "chp"
    if "milliyetci hareket partisi" in party:
        return "mhp"
    if "iyi parti" in party or "i̇yi parti" in party:
        return "iyi"
    if "halklarin esitlik ve demokrasi partisi" in party:
        return "dem"
    return ""

# helper: build up to 10 query variants for a row
def build_query_variants(row):
    # build different query strings for the same person
    name = normalize_text(row.get("name", ""))
    surname = normalize_text(row.get("surname", ""))
    district = normalize_text(row.get("district", ""))
    party = normalize_text(row.get("party", ""))
    pk = party_keyword(row.get("party", ""))

    variants = []

    # 1) full name
    if name and surname:
        variants.append(f"{name} {surname}")
        variants.append(f"{surname} {name}")

    # 2) surname only
    if surname:
        variants.append(surname)

    # 3) name only
    if name:
        variants.append(name)

    # 4) name + surname + district
    if name and surname and district:
        variants.append(f"{name} {surname} {district}")
        variants.append(f"{surname} {district}")
        variants.append(f"{name} {district}")

    # 5) surname + party keyword
    if surname and pk:
        variants.append(f"{surname} {pk}")

    # 6) tbmm hint for deputies
    if bool(row.get("isMilletvekili", False)) and surname:
        variants.append(f"{surname} tbmm")
        variants.append(f"{name} {surname} tbmm")

    # deduplicate while preserving order
    seen = set()
    cleaned = []
    for q in variants:
        q = q.strip()
        if not q:
            continue
        if q not in seen:
            seen.add(q)
            cleaned.append(q)

    # limit to 10 variants at most
    return cleaned[:10]

# helper: score a candidate against a row
def score_candidate(candidate, row):
    # compute a similarity score between candidate and person row
    cand_name = normalize_text(candidate.get("displayName") or "")
    cand_handle = normalize_text(candidate.get("handle") or "")
    full_text = f"{cand_name} {cand_handle}"

    name = normalize_text(row.get("name", ""))
    surname = normalize_text(row.get("surname", ""))
    district = normalize_text(row.get("district", ""))
    party = normalize_text(row.get("party", ""))

    score = 0

    # name match
    if name and name in full_text:
        score += 3
    # surname match (stronger signal)
    if surname and surname in full_text:
        score += 4
    # full name inside display name
    if name and surname and f"{name} {surname}" in cand_name:
        score += 2

    # district hint
    if district and district in full_text:
        score += 1

    # party keyword hint inside handle / name
    pk = party_keyword(party)
    if pk and pk in full_text:
        score += 2

    return score

# helper: call public bluesky search api
def search_bluesky_actor(query, limit=15, timeout=10.0):
    # call bluesky public search api for a given query
    base_url = "https://public.api.bsky.app/xrpc/app.bsky.actor.searchActors"
    params = {"q": query, "limit": limit}
    resp = requests.get(base_url, params=params, timeout=timeout)
    resp.raise_for_status()
    data = resp.json()
    return data.get("actors", []) or []

# multi-pass search
max_passes = 10
min_score = 6  # threshold to accept a match

print("Starting multi-pass Bluesky search on unmatched rows...")

for p in range(max_passes):
    print(f"\nPass {p+1}/{max_passes}...")
    # rows that are not matched yet
    mask_unmatched = df["bsky_match_status"] != "ok"
    # if nothing left, break early
    remaining = mask_unmatched.sum()
    print(f"Remaining unmatched rows before this pass: {remaining}")
    if remaining == 0:
        break

    updated_this_pass = 0

    for idx, row in tqdm(df[mask_unmatched].iterrows(), total=remaining, desc=f"pass {p+1}"):
        variants = build_query_variants(row)
        # if this pass index exceeds variant count, skip this row in this pass
        if p >= len(variants):
            continue

        query = variants[p]

        try:
            candidates = search_bluesky_actor(query, limit=15, timeout=10.0)
        except Exception:
            # skip on any network error, will try other passes
            continue

        best_cand = None
        best_score = -1

        for cand in candidates:
            sc = score_candidate(cand, row)
            if sc > best_score:
                best_score = sc
                best_cand = cand

        if best_cand is not None and best_score >= min_score:
            df.at[idx, "bsky_handle"] = best_cand.get("handle")
            df.at[idx, "bsky_displayName"] = best_cand.get("displayName")
            df.at[idx, "bsky_description"] = best_cand.get("description")
            df.at[idx, "bsky_match_status"] = "ok"
            df.at[idx, "bsky_best_score"] = best_score
            updated_this_pass += 1

        # small sleep to be nice to the api
        time.sleep(0.15)

    # save after each pass
    df.to_csv(csv_path, index=False)
    print(f"Updated rows in this pass: {updated_this_pass}")

print("\nMulti-pass search finished.")
print("Final status summary:")
print(df["bsky_match_status"].value_counts())


Starting multi-pass Bluesky search on unmatched rows...

Pass 1/10...
Remaining unmatched rows before this pass: 1015


pass 1: 100%|██████████| 1015/1015 [05:35<00:00,  3.03it/s]


Updated rows in this pass: 253

Pass 2/10...
Remaining unmatched rows before this pass: 762


pass 2: 100%|██████████| 762/762 [04:18<00:00,  2.95it/s]


Updated rows in this pass: 0

Pass 3/10...
Remaining unmatched rows before this pass: 762


...

Updated rows in this pass: 3

Pass 9/10...
Remaining unmatched rows before this pass: 718


pass 9: 100%|██████████| 718/718 [02:06<00:00,  5.68it/s]


Updated rows in this pass: 0

Pass 10/10...
Remaining unmatched rows before this pass: 718


pass 10: 100%|██████████| 718/718 [00:41<00:00, 17.38it/s]

Updated rows in this pass: 0

Multi-pass search finished.
Final status summary:
bsky_match_status
no_match    718
ok          482
Name: count, dtype: int64


In [22]:
import pandas as pd
import os

base_dir = "/content/drive/MyDrive/Bluesky"

main_path = os.path.join(base_dir, "combined_users_with_bsky.csv")
review_path = os.path.join(base_dir, "no_match_manual_review.csv")

# load main df
df = pd.read_csv(main_path)
print("Loaded main file:", main_path)
print("Shape:", df.shape)

# load manual review df
manual_df = pd.read_csv(review_path)
print("Loaded manual review file:", review_path)
print("Shape:", manual_df.shape)

# check manual handle column
if "bsky_handle_manual" not in manual_df.columns:
    print("\nERROR: Column 'bsky_handle_manual' not found in no_match_manual_review.csv")
    print("Add a column named 'bsky_handle_manual' in that file and fill handles there.")
else:
    # ensure source column exists
    if "bsky_source" not in df.columns:
        df["bsky_source"] = pd.NA

    # mark existing automatic matches
    auto_mask = (df["bsky_match_status"] == "ok") & df["bsky_source"].isna()
    df.loc[auto_mask, "bsky_source"] = "auto"

    updated = 0
    skipped = 0

    # optional: if you also created a displayName manual column, we will use it
    has_manual_name = "bsky_displayName_manual" in manual_df.columns

    # iterate over manual review rows
    for _, row in manual_df.iterrows():
        handle_manual = str(row.get("bsky_handle_manual") or "").strip()
        if not handle_manual:
            skipped += 1
            continue

        pid = row.get("id")
        # find matching row in main df by id
        match = df["id"] == pid
        if not match.any():
            skipped += 1
            continue

        # update main df with manual handle
        df.loc[match, "bsky_handle"] = handle_manual

        # if you defined manual display name, use it, otherwise keep old value
        if has_manual_name:
            disp_manual = str(row.get("bsky_displayName_manual") or "").strip()
            if disp_manual:
                df.loc[match, "bsky_displayName"] = disp_manual

        # status and score for manual matches
        df.loc[match, "bsky_match_status"] = "manual"
        df.loc[match, "bsky_best_score"] = 999
        df.loc[match, "bsky_source"] = "manual"

        updated += int(match.sum())

    # save merged file
    out_path_final = os.path.join(base_dir, "combined_users_with_bsky_final.csv")
    df.to_csv(out_path_final, index=False)

    print(f"\nManual merge finished.")
    print(f"Updated rows from manual review: {updated}")
    print(f"Skipped rows (empty handle or id not found): {skipped}")
    print("\nFinal status summary:")
    print(df["bsky_match_status"].value_counts(dropna=False))
    print("\nOutput saved to:", out_path_final)


Loaded main file: /content/drive/MyDrive/Bluesky/combined_users_with_bsky.csv
Shape: (1200, 18)
Loaded manual review file: /content/drive/MyDrive/Bluesky/no_match_manual_review.csv
Shape: (718, 17)

ERROR: Column 'bsky_handle_manual' not found in no_match_manual_review.csv
Add a column named 'bsky_handle_manual' in that file and fill handles there.


In [23]:
import pandas as pd
import os

base_dir = "/content/drive/MyDrive/Bluesky"
review_path = os.path.join(base_dir, "no_match_manual_review.csv")

df_review = pd.read_csv(review_path)
print("Loaded review file:", review_path)
print("Shape before:", df_review.shape)
print("Columns before:", list(df_review.columns))

# add manual handle/displayName columns if they are missing
if "bsky_handle_manual" not in df_review.columns:
    df_review["bsky_handle_manual"] = ""

if "bsky_displayName_manual" not in df_review.columns:
    df_review["bsky_displayName_manual"] = ""

df_review.to_csv(review_path, index=False)

print("\nSaved updated review file with manual columns.")
print("Shape after:", df_review.shape)
print("Columns after:", list(df_review.columns))
print("\nNow open this file in Google Sheets and fill 'bsky_handle_manual' (and optionally 'bsky_displayName_manual').")


Loaded review file: /content/drive/MyDrive/Bluesky/no_match_manual_review.csv
Shape before: (718, 17)
Columns before: ['id', 'district', 'name', 'surname', 'party', 'alliance', 'isMilletvekili', 'profession', 'title', 'platform', 'political_stance', 'bsky_handle', 'bsky_displayName', 'bsky_match_status', 'bsky_best_score', 'manual_search_query', 'manual_search_url']

Saved updated review file with manual columns.
Shape after: (718, 19)
Columns after: ['id', 'district', 'name', 'surname', 'party', 'alliance', 'isMilletvekili', 'profession', 'title', 'platform', 'political_stance', 'bsky_handle', 'bsky_displayName', 'bsky_match_status', 'bsky_best_score', 'manual_search_query', 'manual_search_url', 'bsky_handle_manual', 'bsky_displayName_manual']

Now open this file in Google Sheets and fill 'bsky_handle_manual' (and optionally 'bsky_displayName_manual').


In [24]:
import pandas as pd
import os

base_dir = "/content/drive/MyDrive/Bluesky"

main_path = os.path.join(base_dir, "combined_users_with_bsky.csv")
review_path = os.path.join(base_dir, "no_match_manual_review.csv")

df = pd.read_csv(main_path)
print("Loaded main file:", main_path)
print("Shape:", df.shape)

manual_df = pd.read_csv(review_path)
print("Loaded manual review file:", review_path)
print("Shape:", manual_df.shape)

if "bsky_handle_manual" not in manual_df.columns:
    print("\nERROR: Column 'bsky_handle_manual' not found in no_match_manual_review.csv")
else:
    if "bsky_source" not in df.columns:
        df["bsky_source"] = pd.NA

    auto_mask = (df["bsky_match_status"] == "ok") & df["bsky_source"].isna()
    df.loc[auto_mask, "bsky_source"] = "auto"

    updated = 0
    skipped = 0
    has_manual_name = "bsky_displayName_manual" in manual_df.columns

    for _, row in manual_df.iterrows():
        handle_manual = str(row.get("bsky_handle_manual") or "").strip()
        if not handle_manual:
            skipped += 1
            continue

        pid = row.get("id")
        match = df["id"] == pid
        if not match.any():
            skipped += 1
            continue

        df.loc[match, "bsky_handle"] = handle_manual

        if has_manual_name:
            disp_manual = str(row.get("bsky_displayName_manual") or "").strip()
            if disp_manual:
                df.loc[match, "bsky_displayName"] = disp_manual

        df.loc[match, "bsky_match_status"] = "manual"
        df.loc[match, "bsky_best_score"] = 999
        df.loc[match, "bsky_source"] = "manual"

        updated += int(match.sum())

    out_path_final = os.path.join(base_dir, "combined_users_with_bsky_final.csv")
    df.to_csv(out_path_final, index=False)

    print(f"\nManual merge finished.")
    print(f"Updated rows from manual review: {updated}")
    print(f"Skipped rows (empty handle or id not found): {skipped}")
    print("\nFinal status summary:")
    print(df['bsky_match_status'].value_counts(dropna=False))
    print("\nOutput saved to:", out_path_final)


Loaded main file: /content/drive/MyDrive/Bluesky/combined_users_with_bsky.csv
Shape: (1200, 18)
Loaded manual review file: /content/drive/MyDrive/Bluesky/no_match_manual_review.csv
Shape: (718, 19)

Manual merge finished.
Updated rows from manual review: 1436
Skipped rows (empty handle or id not found): 0

Final status summary:
bsky_match_status
manual    1018
ok         182
Name: count, dtype: int64

Output saved to: /content/drive/MyDrive/Bluesky/combined_users_with_bsky_final.csv


In [25]:
import pandas as pd
import os

base_dir = "/content/drive/MyDrive/Bluesky"
final_csv = os.path.join(base_dir, "combined_users_with_bsky_final.csv")
out_json = os.path.join(base_dir, "combined_users_with_bsky_final.json")

df = pd.read_csv(final_csv)
print("Loaded:", final_csv)
print("Shape:", df.shape)
print(df[["bsky_match_status", "bsky_source"]].value_counts().head())

# JSON'a koymak istediğin kolonları seç
cols = [
    "id",
    "district",
    "name",
    "surname",
    "party",
    "alliance",
    "status_change",
    "isMilletvekili",
    "profession",
    "title",
    "platform",
    "political_stance",
    "bsky_handle",
    "bsky_displayName",
    "bsky_source",
]

# sadece var olan kolonları al (güvenlik için)
cols = [c for c in cols if c in df.columns]

df[cols].to_json(out_json, orient="records", force_ascii=False, indent=2)

print("\nJSON exported to:", out_json)
print("Columns in JSON:", cols)


Loaded: /content/drive/MyDrive/Bluesky/combined_users_with_bsky_final.csv
Shape: (1200, 19)
bsky_match_status  bsky_source
manual             manual         1018
ok                 auto            182
Name: count, dtype: int64

JSON exported to: /content/drive/MyDrive/Bluesky/combined_users_with_bsky_final.json
Columns in JSON: ['id', 'district', 'name', 'surname', 'party', 'alliance', 'status_change', 'isMilletvekili', 'profession', 'title', 'platform', 'political_stance', 'bsky_handle', 'bsky_displayName', 'bsky_source']


In [28]:
import pandas as pd
import os

# base dir on drive
base_dir = "/content/drive/MyDrive/Bluesky"
main_path = os.path.join(base_dir, "combined_users_with_bsky_final.csv")

# load main csv with current bluesky results
df = pd.read_csv(main_path)
print("Loaded:", main_path)
print("Shape:", df.shape)

# keep only minimal columns for manual editing
cols = ["name", "surname", "party"]
cols = [c for c in cols if c in df.columns]

review_df = df[cols].copy()

# ensure bsky_handle column exists and is string
if "bsky_handle" in df.columns:
    review_df["bsky_handle"] = df["bsky_handle"].fillna("").astype(str)
else:
    review_df["bsky_handle"] = ""

# drop duplicates so you edit each (name, surname, party) once
review_df = review_df.drop_duplicates(subset=["name", "surname", "party"]).reset_index(drop=True)

out_xlsx = os.path.join(base_dir, "bsky_manual_minimal.xlsx")
review_df.to_excel(out_xlsx, index=False)

print("Excel file for manual editing saved to:")
print(out_xlsx)
print("Preview:")
print(review_df.head())


Loaded: /content/drive/MyDrive/Bluesky/combined_users_with_bsky_final.csv
Shape: (1200, 19)
Excel file for manual editing saved to:
/content/drive/MyDrive/Bluesky/bsky_manual_minimal.xlsx
Preview:
       name   surname                       party    bsky_handle
0      Ömer     Çelik  Adalet ve Kalkınma Partisi  omercelik.com
1     Ahmet  Zenbilci                    Bağımsız               
2     Sunay   Karamık                    Bağımsız               
3  Abdullah     Doğru                    Bağımsız               
4     Faruk     Aytek                    Bağımsız               


In [ ]:
import pandas as pd
import os
import numpy as np

# paths
base_dir = "/content/drive/MyDrive/Bluesky"
main_path = os.path.join(base_dir, "combined_users_with_bsky_final.csv")
review_xlsx = os.path.join(base_dir, "bsky_manual_minimal.xlsx")

# load data
main_df = pd.read_csv(main_path)
review_df = pd.read_excel(review_xlsx)

print("Loaded main:", main_path, "shape:", main_df.shape)
print("Loaded review:", review_xlsx, "shape:", review_df.shape)

# ensure required columns exist
for col in ["name", "surname", "party", "bsky_handle"]:
    if col not in review_df.columns:
        raise ValueError(f"Column '{col}' is missing in the Excel file.")

# normalize handle column as string
review_df["bsky_handle"] = review_df["bsky_handle"].fillna("").astype(str).str.strip()

# merge on name + surname + party
merged = main_df.merge(
    review_df,
    on=["name", "surname", "party"],
    how="left",
    suffixes=("", "_manual"),
)

updated = 0

# if manual handle is non-empty, override; otherwise leave original
for idx, row in merged.iterrows():
    manual_handle = row.get("bsky_handle_manual", "")
    if isinstance(manual_handle, str):
        manual_handle = manual_handle.strip()
    else:
        manual_handle = ""

    if manual_handle:
        merged.at[idx, "bsky_handle"] = manual_handle
        merged.at[idx, "bsky_match_status"] = "manual_excel"
        updated += 1
    else:
        # if still empty, mark as no_match_manual (but do not overwrite existing ok/manual)
        status = row.get("bsky_match_status", "")
        if not isinstance(status, str):
            status = ""
        if status.strip() == "" or status in ["no_match", "no_match_manual"]:
            merged.at[idx, "bsky_match_status"] = "no_match_manual"

print("Updated rows from Excel:", updated)

# save updated csv
out_csv = os.path.join(base_dir, "combined_users_with_bsky_edited.csv")
merged.to_csv(out_csv, index=False)

# export compact json for project
json_cols = [
    "id",
    "district",
    "name",
    "surname",
    "party",
    "alliance",
    "status_change",
    "isMilletvekili",
    "profession",
    "title",
    "platform",
    "political_stance",
    "bsky_handle",
]
json_cols = [c for c in json_cols if c in merged.columns]

out_json = os.path.join(base_dir, "combined_users_with_bsky_edited.json")
merged[json_cols].to_json(out_json, orient="records", force_ascii=False, indent=2)

print("CSV saved:", out_csv)
print("JSON saved:", out_json)
print("Final status summary:")
if "bsky_match_status" in merged.columns:
    print(merged["bsky_match_status"].value_counts(dropna=False))
